# 05 Fusion baseline: concatenation, GG1 vs GG2+

Compares three logistic regression models on the same patient-level splits: **MRI only**, **pathology only**, and **MRI + pathology** (concatenated).

**Status:** no real lesion vectors exist yet, so this runs in demo mode on synthetic lesions shaped like the cohort (53 patients, about 3 lesions each, similar grade mix). Demo numbers only confirm the pipeline runs; they are not results.

**Inputs for real data** (all keyed by `lesion_id = irb{IRB}_pat{case}_c{cancer_number}`, e.g. `irb13_pat109_c1`):

| Input | Format | Source |
| --- | --- | --- |
| MRI lesion vectors | CSV: `lesion_id` + feature columns | MRI branch (stand-in: mean and SD of ADC and T2 inside each lesion outline) |
| Pathology lesion vectors | `.npz`: `lesion_id`, `features` (n_lesions × 1024) | Mean of UNI tile vectors per lesion, from the lab's cohort tiles (`pool_tiles` below) |
| Master log | `.xlsx` | Lab spreadsheet with Gleason scores |

**Run on the cluster or locally, not Colab:** the master log contains patient case numbers.

## Settings
Set `DEMO = False` and fill in the three paths once real inputs exist.

In [1]:
DEMO = True
MRI_CSV = "mri_features.csv"
PATH_NPZ = "path_lesions.npz"
MASTER_XLSX = "Master_Log_IRB1318.xlsx"
OUT_CSV = "fusion_baseline_results.csv"

In [2]:
import re
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 0
C_GRID = [0.01, 0.1, 1.0, 10.0]
PATH_PCA = 16   # pathology vectors are reduced so 1,024 numbers don't swamp a few MRI numbers

## Inputs
`load_labels` turns Gleason scores into grade groups and the GG2+ label. `pool_tiles` averages tile vectors into one vector per lesion. `demo_data` builds synthetic lesions. `join` keeps only lesions present in labels, MRI, and pathology, and reports how many were dropped.

In [3]:
def grade_group(text):
    m = re.search(r"(\d)\s*\+\s*(\d)", str(text))
    if not m:
        return None
    p, s = int(m.group(1)), int(m.group(2))
    t = p + s
    return 1 if t <= 6 else (2 if p == 3 else 3) if t == 7 else 4 if t == 8 else 5


def find_col(df, *words):
    for c in df.columns:
        if all(w in " ".join(str(c).lower().split()) for w in words):
            return c
    raise KeyError(f"No column containing {words}: {df.columns.tolist()}")


def load_labels(master_path):
    """lesion_id, patient, grade_group, label (1 = GG2+) from the master log."""
    m = pd.read_excel(master_path)
    irb, case, cancer, gl = (find_col(m, "irb"), find_col(m, "case"),
                             find_col(m, "cancer #"), find_col(m, "gleason"))
    m = m.dropna(subset=[case, cancer])
    to_i = lambda x: int(float(x))
    out = pd.DataFrame({
        "patient": [f"irb{to_i(i)}_pat{to_i(c)}" for i, c in zip(m[irb], m[case])],
        "cancer": m[cancer].map(to_i),
        "grade_group": m[gl].map(grade_group),
    }).dropna(subset=["grade_group"])
    out["lesion_id"] = out.patient + "_c" + out.cancer.astype(str)
    out["label"] = (out.grade_group >= 2).astype(int)
    return out[["lesion_id", "patient", "grade_group", "label"]]


def pool_tiles(tile_features, tile_lesion_ids):
    """Mean of tile vectors per lesion. Returns (lesion_ids, features)."""
    df = pd.DataFrame(tile_features)
    df["lesion_id"] = tile_lesion_ids
    pooled = df.groupby("lesion_id").mean()
    return pooled.index.to_numpy(), pooled.to_numpy()


def demo_data(n_patients=53, seed=SEED):
    """Synthetic lesions shaped like the real cohort, with a weak signal in both modalities."""
    rng = np.random.default_rng(seed)
    rows, mri, path = [], [], []
    for p in range(n_patients):
        for c in range(1, rng.integers(1, 5) + 1):
            gg = int(rng.choice([1, 2, 3, 4, 5], p=[0.30, 0.47, 0.16, 0.02, 0.05]))
            y = int(gg >= 2)
            rows.append((f"irb13_pat{100 + p}_c{c}", f"irb13_pat{100 + p}", gg, y))
            mri.append(rng.normal(0.6 * y, 1, 4))
            path.append(rng.normal(0, 1, 1024) + 0.08 * y)
    labels = pd.DataFrame(rows, columns=["lesion_id", "patient", "grade_group", "label"])
    mri = pd.DataFrame(mri, columns=["adc_mean", "adc_sd", "t2_mean", "t2_sd"])
    mri.insert(0, "lesion_id", labels.lesion_id)
    return labels, mri, labels.lesion_id.to_numpy(), np.array(path)


def join(labels, mri, path_ids, path_feats):
    """Inner join on lesion_id; lesions missing from either modality are dropped and reported."""
    path = pd.DataFrame(path_feats, index=path_ids).add_prefix("path_")
    df = labels.merge(mri, on="lesion_id").merge(path, left_on="lesion_id", right_index=True)
    print(f"{len(labels)} graded lesions, {len(mri)} with MRI, {len(path_ids)} with pathology "
          f"-> {len(df)} lesions in both, {df.patient.nunique()} patients")
    mri_cols = [c for c in mri.columns if c != "lesion_id"]
    path_cols = [c for c in df.columns if c.startswith("path_")]
    return df.reset_index(drop=True), mri_cols, path_cols

## Models and metrics
Pathology vectors are standardized and reduced to 16 principal components (`PATH_PCA`), fitted on training lesions only, so 1,024 pathology numbers don't outweigh a few MRI numbers. All models use L2-regularized logistic regression with balanced class weights.

`run` makes a patient-level 70/15/15 split, tunes C on validation AUC, refits on train + validation, and scores the test lesions. It also reports 5-fold patient-grouped cross-validation, which is more stable with a test set of about 8 patients. **RiskRank** is the share of patients with 2+ lesions whose top-scored lesion has the patient's highest grade group.

In [4]:
def make_model(kind, n_mri, C):
    """Logistic regression on standardized features; pathology reduced with PCA first."""
    lr = LogisticRegression(C=C, class_weight="balanced", max_iter=5000)
    if kind == "mri":
        return make_pipeline(StandardScaler(), lr)
    if kind == "path":
        return make_pipeline(StandardScaler(), PCA(PATH_PCA, random_state=SEED), StandardScaler(), lr)
    # concat: MRI columns come first, pathology columns after
    pre = ColumnTransformer([
        ("mri", StandardScaler(), slice(0, n_mri)),
        ("path", make_pipeline(StandardScaler(), PCA(PATH_PCA, random_state=SEED), StandardScaler()),
         slice(n_mri, None)),
    ])
    return make_pipeline(pre, lr)


def features(df, kind, mri_cols, path_cols):
    cols = {"mri": mri_cols, "path": path_cols, "concat": mri_cols + path_cols}[kind]
    return df[cols].to_numpy()


def risk_ranking(df, scores):
    """Share of patients with 2+ lesions whose top-scored lesion has the patient's highest grade."""
    d = df.assign(score=scores)
    hits = [g.grade_group.loc[g.score.idxmax()] == g.grade_group.max()
            for _, g in d.groupby("patient") if len(g) >= 2]
    return np.mean(hits) if hits else np.nan


def metrics(y, s, df):
    return {"AUC": roc_auc_score(y, s), "AP": average_precision_score(y, s),
            "BalAcc": balanced_accuracy_score(y, s >= 0.5), "F1": f1_score(y, s >= 0.5),
            "RiskRank": risk_ranking(df, s)}


def run(df, mri_cols, path_cols):
    """Patient-level 70/15/15 split: tune C on validation, refit on train + validation, score test.
    Also reports 5-fold patient-grouped cross-validation, which is more stable at this size."""
    y, groups = df.label.to_numpy(), df.patient.to_numpy()
    # 70/15/15 by patient: carve folds of ~15% and take two of them as validation and test
    folds = list(StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED).split(df, y, groups))
    val_idx, test_idx = folds[0][1], folds[1][1]
    train_idx = np.setdiff1d(np.arange(len(df)), np.concatenate([val_idx, test_idx]))
    print(f"split (lesions): train {len(train_idx)}, val {len(val_idx)}, test {len(test_idx)}; "
          f"test positives {y[test_idx].sum()}/{len(test_idx)}")

    rows = []
    for kind in ["mri", "path", "concat"]:
        X = features(df, kind, mri_cols, path_cols)
        best_C = max(C_GRID, key=lambda C: roc_auc_score(
            y[val_idx], make_model(kind, len(mri_cols), C).fit(X[train_idx], y[train_idx])
            .predict_proba(X[val_idx])[:, 1]))
        trval = np.concatenate([train_idx, val_idx])
        s = make_model(kind, len(mri_cols), best_C).fit(X[trval], y[trval]).predict_proba(X[test_idx])[:, 1]
        rows.append({"model": kind, "eval": "test", "C": best_C,
                     **metrics(y[test_idx], s, df.iloc[test_idx])})

        oof = np.zeros(len(df))
        for tr, te in StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(df, y, groups):
            oof[te] = make_model(kind, len(mri_cols), best_C).fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
        rows.append({"model": kind, "eval": "grouped 5-fold CV", "C": best_C, **metrics(y, oof, df)})
    return pd.DataFrame(rows)

## Run

In [5]:
if DEMO:
    labels, mri, pid, pfeat = demo_data()
else:
    labels, mri = load_labels(MASTER_XLSX), pd.read_csv(MRI_CSV)
    z = np.load(PATH_NPZ, allow_pickle=True)
    pid, pfeat = z["lesion_id"], z["features"]

df, mri_cols, path_cols = join(labels, mri, pid, pfeat)
print(f"GG2+ share: {df.label.mean():.2f}")
res = run(df, mri_cols, path_cols)
res.to_csv(OUT_CSV, index=False)
res.round(3)

126 graded lesions, 126 with MRI, 126 with pathology -> 126 lesions in both, 53 patients
GG2+ share: 0.67
split (lesions): train 90, val 19, test 17; test positives 11/17


,model,eval,C,AUC,AP,BalAcc,F1,RiskRank
0,mri,test,0.01,0.773,0.903,0.780,0.800,0.600
1,mri,grouped 5-fold CV,0.01,0.822,0.900,0.766,0.829,0.769
2,path,test,0.01,0.409,0.608,0.485,0.636,0.600
3,path,grouped 5-fold CV,0.01,0.520,0.706,0.547,0.727,0.513
4,concat,test,10.00,0.712,0.851,0.652,0.700,0.400
5,concat,grouped 5-fold CV,10.00,0.811,0.896,0.741,0.819,0.821
